# RAG Evaluation Frameworks

You understand the concepts. Now let's map them to tools — and more importantly, understand *why* each tool exists and what gap it was built to fill.

---

## The Problem Space First

Before comparing frameworks, it helps to see what problem each one was designed around:

| Framework | Core Problem It Was Built To Solve |
|---|---|
| **RAGAS** | I need standardised, reproducible metrics for RAG quality — fast |
| **DeepEval** | I need RAG evaluation that fits into my existing software testing culture |
| **TruLens** | I need to understand what's happening *inside* my pipeline, not just score outputs |
| **LangSmith** | I need observability and evaluation tightly integrated with my LangChain development workflow |

They overlap significantly in features. They differ fundamentally in philosophy.

---

## RAGAS — Standardised Metrics as a Service

### The Problem It Solves

Before RAGAS, every team was rolling their own evaluation metrics — different definitions of "faithfulness," different scoring rubrics, different LLM-as-judge prompts. Results weren't comparable across teams or papers. RAGAS was built to give the RAG community **a shared vocabulary and shared implementation** of the core metrics.

Its central contribution: the **RAG Triad** of metrics, implemented consistently and reproducibly.

### How It Works

RAGAS takes your pipeline's inputs and outputs — questions, retrieved contexts, generated answers, and optionally ground truth answers — and computes standardised scores.

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy,
                          context_recall, context_precision

results = evaluate(
    dataset=your_eval_dataset,
    metrics=[faithfulness, answer_relevancy,
             context_recall, context_precision]
)
```

In a procurement context, you'd feed it:

```python
{
  "question": "Does Vendor B meet the on-time completion requirement?",
  "contexts": [retrieved_chunks],          # what retrieval returned
  "answer": system_generated_answer,       # what LLM produced
  "ground_truth": gold_answer_from_dataset # what the correct answer is
}
```

And get back scores for each metric, each defined consistently:

- **Faithfulness** — what fraction of claims in the answer are supported by the retrieved contexts?
- **Answer Relevancy** — does the answer actually address the question asked?
- **Context Recall** — does the retrieved context contain what's needed to answer?
- **Context Precision** — is the retrieved context free of irrelevant noise?

### What Makes It Useful

RAGAS lets you **benchmark across system versions with confidence** that you're measuring the same thing each time. When you switch your chunking strategy, change your embedding model, or upgrade your LLM, you run the same RAGAS suite and get comparable numbers.

It's also reference-free for several metrics — faithfulness and answer relevancy can be computed without ground truth answers, using only the question, context, and generated answer. This matters in production where you often don't have gold answers for every query.

### What It Doesn't Solve

RAGAS scores your RAG system as a black box. It tells you that faithfulness is 0.72 — it doesn't tell you *which* pipeline stage caused the failures, or *why* a particular answer was unfaithful. It's a measuring instrument, not a diagnostic tool.

It also has no concept of your workflow. It doesn't integrate with your CI/CD pipeline, it doesn't alert you when scores drop, and it doesn't help you track evaluation over time across deployments.

### When to Use It

When you need a credible, standardised benchmark — comparing your system to published baselines, communicating quality to stakeholders, or running A/B comparisons between system variants in your procurement RAG.

---

## DeepEval — RAG Evaluation as Software Testing

### The Problem It Solves

RAGAS gives you metrics. DeepEval gives you **a testing framework** — one that software engineers already know how to use.

The insight behind DeepEval: most teams building RAG systems already have engineers who write unit tests, run CI/CD pipelines, and expect tests to pass or fail with clear messages. RAGAS produces a score. DeepEval produces a **test result** — pass, fail, with a reason.

### How It Works

DeepEval wraps RAG metrics in pytest-compatible test cases:

```python
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

def test_vendor_eligibility_answer():
    test_case = LLMTestCase(
        input="Does Vendor B meet the on-time completion requirement?",
        actual_output=system_generated_answer,
        retrieval_context=retrieved_chunks,
        expected_output=gold_answer
    )

    assert_test(test_case, [
        FaithfulnessMetric(threshold=0.8),
        AnswerRelevancyMetric(threshold=0.75)
    ])
```

You run this with `pytest`. If faithfulness drops below 0.8, the test fails. Your CI pipeline blocks the deployment. The engineer gets a clear failure message explaining which metric failed and why.

This is the key difference: RAGAS tells you a score. DeepEval tells you whether you should ship.

### What Makes It Useful

**Regression prevention.** You make a change to your chunking strategy — maybe you reduce chunk size from 500 to 300 tokens to improve precision. DeepEval runs your full test suite and tells you that context recall dropped from 0.84 to 0.71 on your procurement evaluation cases. You catch this before deployment rather than after a procurement officer reports worse answers.

**Threshold enforcement.** For high-stakes systems like a procurement award recommendation engine, you can set strict thresholds — faithfulness must be above 0.85, answer relevancy above 0.80 — and those become hard gates on deployment.

**Red-teaming.** DeepEval includes adversarial test cases — inputs designed to break your system. For procurement RAG, you can test whether the system can be manipulated into recommending a debarred vendor, or whether it handles contradictory information across documents correctly.

### What It Doesn't Solve

DeepEval is fundamentally a **pre-deployment testing tool**. It's excellent at catching regressions before you ship. It's not built for production monitoring — it doesn't watch your live system and alert you when answer quality degrades in the field.

It also requires you to maintain an eval dataset of test cases. The framework is powerful, but only as powerful as the test cases you've written.

### When to Use It

When your team has engineering discipline and wants RAG quality to be a first-class part of your development process — treated the same way as unit tests and integration tests, with the same CI/CD integration and the same pass/fail culture.

---

## TruLens — Tracing the Inside of Your Pipeline

### The Problem It Solves

Both RAGAS and DeepEval treat your RAG system as a box: question goes in, answer comes out, metrics are computed on the output. TruLens was built around a different question: **what happened inside the box?**

TruLens is fundamentally an **instrumentation and tracing tool** that happens to include evaluation. It wraps your pipeline components, records every intermediate step — what the retriever returned, what the reranker did, what each tool call produced — and then evaluates quality at each step independently.

### How It Works

TruLens uses a decorator pattern to instrument your pipeline:

```python
from trulens.apps.langchain import TruChain
from trulens.core import TruSession
from trulens.providers.openai import OpenAI

session = TruSession()
provider = OpenAI()

# Define feedback functions (TruLens term for evaluation metrics)
f_faithfulness = (
    Feedback(provider.groundedness_measure_with_cot_reasons)
    .on(Select.RecordCalls.retriever.get_relevant_documents.rets)
    .on_output()
)

# Wrap your pipeline
tru_pipeline = TruChain(
    your_rag_chain,
    feedbacks=[f_faithfulness, f_answer_relevance, f_context_relevance]
)

# Every call is now traced and evaluated
with tru_pipeline as recording:
    response = tru_pipeline.invoke(
        "Does Vendor B meet the on-time completion requirement?"
    )
```

After running queries through your instrumented pipeline, TruLens provides a dashboard showing every execution trace — what the retriever returned at step 2, what the reranker changed at step 3, what the LLM produced at step 4 — with evaluation scores attached at each stage.

### The RAG Triad in TruLens

TruLens popularised the framing of evaluation as three questions, applied to specific relationships in your pipeline:

```
Context Relevance:    Question ←→ Retrieved Context
                      (Did retrieval return relevant chunks?)

Groundedness:         Retrieved Context ←→ Generated Answer
                      (Did the LLM stay within the context?)

Answer Relevance:     Question ←→ Generated Answer
                      (Did the answer address what was asked?)
```

The key insight: these three relationships let you localise failures. If context relevance is low but groundedness is high, retrieval is your problem. If context relevance is high but groundedness is low, your LLM is hallucinating despite good inputs.

### What Makes It Useful

**Pipeline-level debugging.** When a procurement query produces a wrong answer, TruLens shows you the full execution trace. You can see that the retriever returned the right documents, the reranker correctly ranked them, but the LLM ignored the on-time completion rate in favour of the overall project count. That's a generation problem, not a retrieval problem — and you wouldn't know that from an output-level score alone.

**Production monitoring.** TruLens is designed to run on live traffic, not just eval datasets. You instrument your production pipeline, and it continuously evaluates every real user query. You see quality trends over time — if faithfulness starts declining after a model update, you catch it in hours, not weeks.

**Experiment comparison.** Run the same queries through two different pipeline configurations — different chunk sizes, different retrievers, different models — and compare their traces side by side.

### What It Doesn't Solve

TruLens requires more instrumentation effort than RAGAS or DeepEval. You need to wrap your pipeline components, and the tracing overhead adds latency to your evaluations. For teams that just want a quick quality benchmark, it's more setup than necessary.

It also doesn't have the software-testing integration that DeepEval provides. There's no native pytest runner, no CI/CD gate, no threshold-based pass/fail on deployment.

### When to Use It

When you're debugging a complex pipeline and need to understand *where* quality is being lost — not just *how much* quality is lost. Also when you need production monitoring: continuous quality tracking on real user queries rather than just on a fixed eval dataset.

---

## LangSmith — Evaluation Inside Your Development Environment

### The Problem It Solves

RAGAS, DeepEval, and TruLens are evaluation tools you bring *to* your RAG system. LangSmith is evaluation built *into* your development environment — specifically, the LangChain ecosystem.

The problem it solves: **context switching.** When you're building a RAG pipeline in LangChain and want to evaluate it, you previously had to export data to a separate eval tool, run it there, interpret results separately, then come back to your code. LangSmith makes evaluation a native part of the LangChain development loop.

### How It Works

LangSmith provides three things in an integrated package:

**1. Automatic tracing.** Set an environment variable and every LangChain run is automatically traced — every chain, every retriever call, every LLM call, every tool invocation. No instrumentation code required.

```python
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your_key"

# Everything below is automatically traced
response = your_rag_chain.invoke(
    "Can we award the civil works contract to the L1 bidder?"
)
```

Every execution appears in the LangSmith UI — full trace, every intermediate value, latency at each step, token counts, cost.

**2. Dataset management and evaluation.** You build your eval dataset in LangSmith, then run evaluations against it:

```python
from langsmith.evaluation import evaluate

results = evaluate(
    your_rag_chain,
    data="procurement-eval-dataset",
    evaluators=[correctness_evaluator, faithfulness_evaluator],
    experiment_prefix="chunking-strategy-v2"
)
```

Results are stored against the named experiment. When you run the same eval with a different configuration — say, after changing your embedding model — you get a direct comparison in the UI.

**3. Human feedback integration.** LangSmith makes it straightforward to collect human feedback on traces — a reviewer can look at a trace, see what the system retrieved and generated, and annotate it directly. This human feedback feeds back into your eval dataset.

### What Makes It Useful

**Zero-friction tracing.** If you're building in LangChain, you get complete pipeline visibility with one environment variable. For teams already in the LangChain ecosystem, this is the path of least resistance to understanding what their pipeline is doing.

**Experiment management.** LangSmith tracks every named experiment — you can look at your last 8 configuration changes and see exactly how each metric moved. This is the kind of experiment history that teams otherwise maintain in spreadsheets.

**The development loop.** The real value is how LangSmith collapses the feedback loop: you write code, run a query, see the full trace in the UI immediately, identify what went wrong, fix it, run again. Evaluation becomes part of writing code rather than a separate phase.

### What It Doesn't Solve

LangSmith is tightly coupled to LangChain. If your pipeline uses a different orchestration framework — LlamaIndex, Haystack, a custom implementation — LangSmith's automatic tracing doesn't apply. You can still use it, but you lose the zero-friction advantage and need to instrument manually.

It's also a **commercial SaaS product** with pricing that scales with usage. At high query volumes in a production procurement system, costs need to be evaluated alongside the value.

The evaluation metrics themselves are also less standardised than RAGAS. LangSmith gives you the infrastructure to *run* evaluations, but the quality of those evaluations depends on the evaluators you write or configure — there's no canonical procurement-RAG metric set out of the box.

### When to Use It

When your team is building in LangChain and wants evaluation to feel like a native part of development rather than a separate tool. Also when experiment tracking and human feedback collection are as important as the metric scores themselves.

---

## Direct Comparison

| Dimension | RAGAS | DeepEval | TruLens | LangSmith |
|---|---|---|---|---|
| **Core philosophy** | Standardised metrics | Testing framework | Pipeline instrumentation | Development environment |
| **Primary output** | Scores | Pass / Fail | Traces + scores | Traces + experiments |
| **Stage visibility** | Output only | Output only | Full pipeline | Full pipeline |
| **Production monitoring** | No | No | Yes | Yes |
| **CI/CD integration** | Manual | Native (pytest) | No | Partial |
| **Framework coupling** | None | None | Light | Heavy (LangChain) |
| **Setup effort** | Low | Low–Medium | Medium–High | Low (in LangChain) |
| **Standardisation** | High | Medium | Medium | Low |
| **Human feedback** | No | No | Limited | Yes |
| **Best for** | Benchmarking | Regression testing | Debugging & monitoring | LangChain dev loop |

---

## How They Fit Together in Practice

These tools aren't mutually exclusive. Mature teams often combine them:

```
Development phase
  └── LangSmith traces every experiment
  └── RAGAS benchmarks each configuration change

Pre-deployment
  └── DeepEval runs the test suite in CI
  └── Blocks deployment if thresholds fail

Production
  └── TruLens monitors live queries continuously
  └── Alerts when faithfulness or completeness degrades
  └── Human reviewers annotate flagged traces in LangSmith

Quarterly review
  └── RAGAS re-benchmarks against updated eval dataset
  └── Results compared to previous quarters for trend analysis
```

For a procurement RAG system specifically, the highest-priority combination is usually **DeepEval for deployment gates** (because award recommendations carry legal weight and you cannot afford regressions) and **TruLens for production monitoring** (because you need to catch quality degradation on real queries, not just on your eval dataset).

RAGAS is most useful when you need to communicate quality externally — to a procurement committee, a CISO, or a vendor who wants to audit your system. The standardised metrics give you a defensible, reproducible number.

LangSmith pays off most if your team is already in LangChain and values the development-loop integration over the portability of a framework-agnostic tool.